In [0]:
# Change the course name as per your course name
COURSE_NAME = "Data Interoperability with Unity Catalog"
RUN_QA_CHECKER = True
QA_TASK_RELATIVE_PATH = "./qa_content_checker"
QA_TASK_NAME = "QA Content Checker — Grammar, Deprecated, UI Steps"

# Tester emails — notified on job failure and success. Add as many as you like with comma seperated emails
TESTER_EMAILS = [
    "gourav.prajapati@databricks.com"
]

# Add your lab notebook paths relative from the CourseRunner folder as per your course.
# Only executable labs are listed here.
LAB_NOTEBOOKS = [
    "../../3. Centralized Data Processing with External Analytics/3.3 Lab - Iceberg Read Access on UC Managed Tables",
    "../../5. Lakehouse Federation/5.6 Lab - Implementing Lakehouse Federation",
]

# ── Compute Type Registry ─────────────────────────────────────────────────────
DEFAULT_COMPUTE_TYPE = "serverless_v5"

COMPUTE_TYPES = {
    "serverless_v5": {
        "type": "serverless",
        "environment_version": "5",
    },
    "shared_warehouse": {
        "type": "warehouse",
        "warehouse_name": "shared_warehouse",
        "cluster_size": "2X-Small",
        "warehouse_type": "PRO",
        "auto_stop_mins": 30,
    },
}

# ── Task Retry Policy ─────────────────────────────────────────────────────────
TASK_MAX_RETRIES = 2                    # Number of retries on failure (0 = no retries)
TASK_RETRY_INTERVAL_SECONDS = 30        # Seconds to wait between retry attempts (global default)
WAREHOUSE_RETRY_INTERVAL_SECONDS = 45   # Seconds for warehouse tasks (cross-compute metadata sync is slower)

# ── Course notebook DAG ───────────────────────────────────────────────────────
COURSE_TASKS = [
    {
        "task_key": "required_setup",
        "name": "0 - Required Setup",
        "relative_path": "../../0 - Required Setup",
        "depends_on": [],
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "instructor_demo_setup",
        "name": "1 - Instructor Demo Setup",
        "relative_path": "../../1 - Instructor Demo Setup",
        "depends_on": ["required_setup"],
        "compute_type": "serverless_v5",
    },
    # ── Module 2 ──
    {
        "task_key": "m02_demo",
        "name": "2.2 Demo - Interoperability and Performance Benefits of Managed Tables",
        "relative_path": "../../2. Working with Managed Tables in Unity Catalog/2.2 Demo - Interoperability and Performance Benefits of Managed Tables",
        "depends_on": ["instructor_demo_setup"],
        "compute_type": "serverless_v5",
    },
    # ── Module 3 ──
    # Video Demonstration
    # This demo requires a Snowflake account, Databricks OAuth service principal,
    # and customer-managed storage, which are not available in the lab environment.
    # Therefore, only the Classroom Setup notebook is included here. The complete
    # demo is provided as a video walkthrough for environments that meet these prerequisites.
    {
        "task_key": "m03_demo_Classroom_Setup",
        "name": "Classroom-Setup-3-demo",
        "relative_path": "../Classroom-Setup-3-demo",
        "depends_on": ["m02_demo"],
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "m03_lab",
        "name": "3.3 Lab - Iceberg Read Access on UC Managed Tables",
        "relative_path": "../../3. Centralized Data Processing with External Analytics/3.3 Lab - Iceberg Read Access on UC Managed Tables",
        "depends_on": ["m03_demo_Classroom_Setup"],
        "compute_type": "serverless_v5",
    },
    # ── Module 4 ──
    # Video Demonstration
    # This demo requires an AWS account with EMR permissions, Databricks OAuth service principal,
    # and customer-managed S3 storage, which are not available in the lab environment.
    # Therefore, the complete demo is provided as a video walkthrough. The notebook contains
    # Databricks SQL execution and AWS-side reference code for environments with prerequisites.
    {
        "task_key": "m04_demo_Classroom_Setup",
        "name": "Classroom-Setup-4",
        "relative_path": "../Classroom-Setup-4",
        "depends_on": ["m03_lab"],
        "compute_type": "serverless_v5",
    },
    # ── Module 5 ──
    # Video Demonstration
    # This demo requires external source systems and network connectivity, which are not
    # available in the lab environment. The complete demo is provided as a video walkthrough.
    # The notebook contains the working demo for environments with the required infrastructure.
    #
    # Required Permissions:
    # - Databricks: Metastore Admin or CREATE CONNECTION privilege.
    # - SQL Server: Running SQL Server with AdventureWorksDW restored and read access.
    # - Snowflake: ACCOUNTADMIN or privileges to create databases and grant access.
    {
        "task_key": "m05_demo1__Classroom_Setup",
        "name": "5.2 Demo - Federation to SQL Server and Snowflake",
        "relative_path": "../Classroom-Setup-Common",
        "depends_on": ["m04_demo_Classroom_Setup"],
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "m05_demo_2_Classroom_Setup",
        "name": "5.4 Demo - Creating a Foreign Catalog Connection to AWS Glue Catalog",
        "relative_path": "../Classroom-Setup-Common",
        "depends_on": ["m05_demo1__Classroom_Setup"],
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "m05_demo_3_Classroom_Setup",
        "name": "5.5 Demo - Implementing Lakehouse Federation",
        "relative_path": "../Classroom-Setup-5",
        "depends_on": ["m05_demo_2_Classroom_Setup"],
        "compute_type": "serverless_v5",
    },
    # Due to SET VAR pg_host_value = '{lakebase-host}';
    # SET VAR pg_oauth_token_value = '{oauth-token}'; these commands can only be run in the Classroom setup.
    {
        "task_key": "m05_lab_Classroom_Setup",
        "name": "5.6 Lab - Implementing Lakehouse Federation",
        "relative_path": "../Classroom-Setup-5-lab",
        "depends_on": ["m05_demo_3_Classroom_Setup"],
        "compute_type": "serverless_v5",
    },
]

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: wrap_failing_sql_cell / restore_sql_cell
#
# Purpose
# -------
# Some demo notebooks contain SQL cells that are *intentionally designed to fail*
# (e.g. to demonstrate BEGIN...END halting behaviour). When those notebooks run
# as job tasks the cell failure stops the entire notebook, skipping all
# subsequent cells (cleanup, further demos, etc.).
#
# These helpers let the course tester temporarily wrap any such cell with an
# outer BEGIN...DECLARE EXIT HANDLER...END block — pure SQL, no language change
# — so the notebook continues past the intentional failure. After the job
# completes the original source is restored so students always see the clean
# demo notebook.
#
# Identifying the target cell
# ----------------------------
# Cells are identified by cell_index: the 1-based position of the cell within
# the notebook (counting ALL cells, markdown included). Unlike cell_nuid —
# which regenerates every time a lab notebook is re-provisioned/reset — a
# cell's position in an unchanged template stays the same across resets, so
# cell_index remains valid as long as no cells are added/removed above the
# target cell.
#
# How to use
# ----------
# 1. Before job creation / run:
#       restores = patch_notebooks_for_job(CELLS_TO_PATCH)
#
# 2. After job completes (pass/fail):
#       restore_notebooks_after_job(restores)
#
# CELLS_TO_PATCH is a list of dicts defined in the cell that follows this one.
# Each entry:
#   {
#       "notebook_path"  : absolute workspace path to the notebook,
#       "cell_index"     : 1-based position of the SQL cell to wrap,
#       "handler_message": (optional) text returned by the handler SELECT,
#   }
# ─────────────────────────────────────────────────────────────────────────────

import base64, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

_sdk_client = None

def _get_client(w=None):
    """Returns a WorkspaceClient, reusing a module-level instance if available."""
    if w:
        return w
    global _sdk_client
    if _sdk_client is None:
        _sdk_client = WorkspaceClient()
    return _sdk_client


def _get_cell_source(cell):
    """Returns a cell's source as a single string, regardless of list/str form."""
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source


def wrap_failing_sql_cell(
    notebook_path: str,
    cell_index: int,
    handler_message: str = None,
    w=None,
) -> str:
    """
    Wraps a SQL cell's BEGIN...END block with an outer DECLARE EXIT HANDLER
    so the notebook continues executing even if the inner block fails.
    The cell stays SQL — no language change.

    Args:
        notebook_path   : Absolute workspace path to the target notebook.
        cell_index      : 1-based position of the cell to wrap (counting ALL
                          cells, markdown included).
        handler_message : Optional message shown as a result row when the
                          exception is caught.
        w               : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — pass this to restore_sql_cell() to undo.
    """
    _w  = _get_client(w)
    msg = handler_message or (
        "[Expected failure] The inner BEGIN...END block failed as designed. "
        "Notebook continues."
    )

    # --- export
    export  = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb      = json.loads(base64.b64decode(export.content))
    cells   = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cell = cells[cell_index - 1]

    original_source = _get_cell_source(cell)
    indented = "\n".join("  " + line for line in original_source.splitlines())
    cell["source"] = (
        "-- Outer block catches the expected error; inner block is unchanged.\n"
        "BEGIN\n"
        "  DECLARE EXIT HANDLER FOR SQLEXCEPTION\n"
        f"    SELECT '{msg}' AS message;\n\n"
        "  BEGIN\n"
        f"{indented}\n"
        "  END;\n"
        "END;"
    )

    # --- re-import
    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Wrapped  : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_sql_cell(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses wrap_failing_sql_cell().
    Use the SAME cell_index that was passed to wrap_failing_sql_cell().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by wrap_failing_sql_cell().
        w              : WorkspaceClient to reuse (optional).
    """
    _w     = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb     = json.loads(base64.b64decode(export.content))
    cells  = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_notebooks_for_job(patches: list, w=None) -> list:
    """
    Wraps all cells listed in `patches` before a job run.
    Returns a restore-list to pass to restore_notebooks_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path   (str) — absolute workspace path
                    cell_index      (int) — 1-based position of the failing cell
                    handler_message (str, opt) — custom handler message
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    print("\n═" * 36 + "\nPATCHING NOTEBOOKS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = wrap_failing_sql_cell(
            notebook_path   = p["notebook_path"],
            cell_index      = p["cell_index"],
            handler_message = p.get("handler_message"),
            w               = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_notebooks_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_notebooks_for_job().

    Args:
        restores : the list returned by patch_notebooks_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    print("\n═" * 36 + "\nRESTORING NOTEBOOKS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_sql_cell(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")



# ─────────────────────────────────────────────────────────────────────────────
# replace_cell_text / restore_cell_text
#
# Purpose
# -------
# Generic text-replacement helpers: swap a specific substring in a notebook
# cell's source before a job run, then put it back afterward.
#
# Useful when a single line in an already-filled notebook needs a small tweak
# for automated testing (e.g. converting an intentional single-statement
# failure into a no-op) without wrapping the whole cell in a
# BEGIN...DECLARE EXIT HANDLER block.
#
# Four parameters
# ---------------
#   notebook_path    — absolute workspace path to the notebook
#   cell_index       — 1-based position of the cell (counting ALL cells)
#   original_text    — the exact text to find (can span multiple lines)
#   replacement_text — the text to substitute in
# ─────────────────────────────────────────────────────────────────────────────

def replace_cell_text(
    notebook_path: str,
    cell_index: int,
    original_text: str,
    replacement_text: str,
    w=None,
) -> str:
    """
    Replaces a specific text substring in a notebook cell's source.

    Args:
        notebook_path    : Absolute workspace path to the target notebook.
        cell_index       : 1-based position of the cell to modify (counting ALL cells).
        original_text    : The exact text to find and replace.
        replacement_text : The text to substitute in.
        w                : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — the full original cell source before replacement.
                                Pass to restore_cell_text() to undo.
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )

    cell = cells[cell_index - 1]
    original_source = _get_cell_source(cell)

    if original_text not in original_source:
        raise ValueError(
            f"original_text not found in cell {cell_index} of "
            f"{notebook_path.split('/')[-1]!r}.\n"
            f"original_text: {original_text!r}"
        )

    cell["source"] = original_source.replace(original_text, replacement_text, 1)

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Replaced : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_cell_text(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses replace_cell_text().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by replace_cell_text().
        w              : WorkspaceClient to reuse (optional).
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_text_replacements_for_job(patches: list, w=None) -> list:
    """
    Applies targeted text replacements to notebook cells before a job run.
    Returns a restore-list to pass to restore_text_replacements_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path    (str) — absolute workspace path
                    cell_index       (int) — 1-based position of the cell
                    original_text    (str) — exact text to replace
                    replacement_text (str) — text to substitute in
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    if not patches:
        print("  (no text replacements configured)\n")
        return []
    print("\n" + "═" * 36 + "\nAPPLYING TEXT REPLACEMENTS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = replace_cell_text(
            notebook_path    = p["notebook_path"],
            cell_index       = p["cell_index"],
            original_text    = p["original_text"],
            replacement_text = p["replacement_text"],
            w                = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_text_replacements_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_text_replacements_for_job().

    Args:
        restores : the list returned by patch_text_replacements_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    if not restores:
        print("  (nothing to restore)\n")
        return
    print("\n" + "═" * 36 + "\nRESTORING TEXT REPLACEMENTS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_cell_text(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")


print("✅ Utility functions loaded: wrap_failing_sql_cell, restore_sql_cell,"
      " patch_notebooks_for_job, restore_notebooks_after_job,"
      " replace_cell_text, restore_cell_text,"
      " patch_text_replacements_for_job, restore_text_replacements_after_job")